In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Загружаем данные заново
url = "https://raw.githubusercontent.com/selva86/datasets/master/Cars93_miss.csv"
df = pd.read_csv(url)

# 2. Быстрая очистка (как в ЛР №1), чтобы получить df_cleaned
def quick_clean(data):
    df_copy = data.copy()
    numeric_cols = df_copy.select_dtypes(include=[np.number]).columns
    df_copy[numeric_cols] = df_copy[numeric_cols].fillna(df_copy[numeric_cols].median())
    object_cols = df_copy.select_dtypes(include=['object']).columns
    df_copy[object_cols] = df_copy[object_cols].fillna('Unknown')
    return df_copy

df_cleaned = quick_clean(df) # Теперь переменная определена!

In [3]:
from sklearn.preprocessing import LabelEncoder

# 1. Label Encoding для колонки 'Cylinders' (т.к. там есть порядок: 3, 4, 6 цилиндров)
le = LabelEncoder()
df_cleaned['Cylinders_Encoded'] = le.fit_transform(df_cleaned['Cylinders'].astype(str))

# 2. One-Hot Encoding для 'Type' (тип кузова)
# Используем pd.get_dummies — это стандартный и быстрый способ в анализе данных
df_encoded = pd.get_dummies(df_cleaned, columns=['Type'], prefix='Type')

print("Категориальные признаки закодированы.")
df_encoded[['Cylinders_Encoded', 'Type_Midsize', 'Type_Small']].head()

Категориальные признаки закодированы.


,Cylinders_Encoded,Type_Midsize,Type_Small
0,1,False,True
1,3,True,False
2,3,False,False
3,3,True,False
4,1,True,False


Использование LabelEncoder и One-Hot Encoding для преобразования текста в числа

In [5]:
from sklearn.preprocessing import MinMaxScaler

# Создаем 2 новых признака (как просит ТЗ)
df_encoded['HP_per_EngineSize'] = df_encoded['Horsepower'] / df_encoded['EngineSize']
df_encoded['Weight_to_Wheelbase'] = df_encoded['Weight'] / df_encoded['Wheelbase']

# Масштабирование признаков в диапазон [0, 1]
scaler = MinMaxScaler()
cols_to_scale = ['Price', 'Horsepower', 'Weight', 'EngineSize', 'HP_per_EngineSize']

# Применяем масштабирование
df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])

print("Масштабирование выполнено. Значения теперь в диапазоне [0, 1].")

Масштабирование выполнено. Значения теперь в диапазоне [0, 1].


Применение MinMaxScaler для нормализации данных и создание признаков HP_per_EngineSize и Weight_to_Wheelbase

In [6]:
# Список колонок для удаления (текстовые идентификаторы)
# Обоснование: Уникальные названия моделей и брендов могут привести к переобучению
to_drop = ['Model', 'Manufacturer', 'Make', 'Cylinders'] # Удаляем старый Cylinders, т.к. есть Encoded

df_final = df_encoded.drop(columns=[c for c in to_drop if c in df_encoded.columns])

print("Финальный датасет готов!")
print("Количество признаков:", df_final.shape[1])
df_final.head()

Финальный датасет готов!
Количество признаков: 32


,Min.Price,Price,Max.Price,MPG.city,MPG.highway,AirBags,DriveTrain,EngineSize,Horsepower,RPM,...,Cylinders_Encoded,Type_Compact,Type_Large,Type_Midsize,Type_Small,Type_Sporty,Type_Unknown,Type_Van,HP_per_EngineSize,Weight_to_Wheelbase
0,12.9,0.155963,18.80,25.0,31.0,Unknown,Front,0.170213,0.346939,6300.0,...,1,False,False,False,True,False,False,False,0.125591,0.004109
1,29.2,0.486239,38.70,18.0,25.0,Driver & Passenger,Front,0.468085,0.591837,5500.0,...,3,False,False,True,False,False,False,False,0.062648,0.006729
2,25.9,0.398165,32.30,20.0,26.0,Driver only,Front,0.382979,0.477551,5500.0,...,3,True,False,False,False,False,False,False,0.061229,0.006834
3,14.6,0.555963,44.60,19.0,26.0,Driver & Passenger,Unknown,0.276596,0.477551,5500.0,...,3,False,False,True,False,False,False,False,0.100236,0.006694
4,14.6,0.414679,19.15,22.0,30.0,Unknown,Rear,0.531915,0.624490,5700.0,...,1,False,False,True,False,False,False,False,0.055300,0.007404


Удаление нерелевантных признаков (Model, Manufacturer), так как они являются строковыми идентификаторами